In [1]:
import json
import os
import pandas as pd
import numpy as np

In [2]:
# The five elements used in the FeCoCrNiNbx alloy family
ELEMENTS = ["Fe", "Co", "Cr", "Ni", "Nb"]

# Atomic masses in g/mol
ATOMIC_MASS = {"Fe": 55.845, "Co": 58.933, "Cr": 51.996, "Ni": 58.693, "Nb": 92.906}

In [3]:
def load_data(path="laser_cladding_data.csv"):
    """Read the experiment table from a CSV file."""
    return pd.read_csv(path)

In [4]:
data = load_data()

In [5]:
data.head(10)

,id,source,substrate,composition,nb_content,laser_power_W,scan_speed_mm_s,porosity,crack,outcome
0,1,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,2.00,1,0,partial
1,2,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,1.72,1,0,partial
2,3,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,1.77,0,0,good
3,4,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,1.67,1,0,partial
4,5,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,3.00,1,0,partial
5,6,own_thesis,AISI420,FeCoCrNiNb0.15,0.15,280,1.62,1,0,failed
6,7,own_thesis,AISI420,FeCoCrNiNb0.25,0.25,280,2.10,0,0,good
7,8,own_thesis,AISI420,FeCoCrNiNb0.25,0.25,280,3.00,0,0,good
8,9,own_thesis,AISI420,FeCoCrNiNb0.25,0.25,280,4.00,0,0,good
9,10,own_thesis,AISI420,FeCoCrNiNb0.25,0.25,280,4.00,0,0,good


In [6]:
def validate(proposal):
    """Check a proposed composition (dict of molar fractions).
    Returns (True, "OK") or (False, reason)."""
    for el in proposal:
        if el not in ELEMENTS:
            return False, f"Element '{el}' is not allowed"
    for el in ELEMENTS:
        if el not in proposal:
            return False, f"Element '{el}' is missing"
    if any(v < 0 for v in proposal.values()):
        return False, "Fractions cannot be negative"
    total = sum(proposal.values())
    if abs(total - 1) > 0.001:
        return False, f"Fractions sum to {total:.3f}, not 1"
    return True, "OK"

In [7]:
# Quick tests
print(validate({"Fe": 0.2, "Co": 0.2, "Cr": 0.2, "Ni": 0.2, "Nb": 0.2}))   # valid
print(validate({"Fe": 0.3, "Co": 0.3, "Cr": 0.3, "Ni": 0.3, "Nb": 0.3}))   # sum is 1.5
print(validate({"Fe": 0.2, "Co": 0.2, "Cr": 0.2, "Ni": 0.2, "Ti": 0.2}))   # Ti not allowed

(True, 'OK')
(False, 'Fractions sum to 1.500, not 1')
(False, "Element 'Ti' is not allowed")


In [8]:
def powder_mass(proposal, total_g):
    """Mass in grams of each element powder for a batch of total_g grams."""
    weights = {el: frac * ATOMIC_MASS[el] for el, frac in proposal.items()}
    total = sum(weights.values())
    return {el: round(w / total * total_g, 3) for el, w in weights.items()}

In [9]:
example = {"Fe": 0.2, "Co": 0.2, "Cr": 0.2, "Ni": 0.2, "Nb": 0.2}
print(powder_mass(example, 10))

{'Fe': 1.754, 'Co': 1.851, 'Cr': 1.633, 'Ni': 1.844, 'Nb': 2.918}


In [10]:
def nb_ratio(proposal):
    """Convert molar fractions to the x in FeCoCrNiNbx."""
    others = [proposal[el] for el in ["Fe", "Co", "Cr", "Ni"]]
    return proposal["Nb"] / (sum(others) / 4)

In [11]:
def suggest_params(proposal, data):
    """Find past experiments with the closest Nb content and summarize them."""
    x = nb_ratio(proposal)
    closest_idx = (data["nb_content"] - x).abs().idxmin()
    nearest = data.loc[closest_idx, "nb_content"]
    subset = data[data["nb_content"] == nearest]
    good = subset[subset["outcome"] == "good"]
    return {
        "requested_nb": round(float(x), 2),
        "nearest_nb_in_data": float(nearest),
        "in_data_range": bool(data["nb_content"].min() <= x <= data["nb_content"].max()),
        "n_similar_experiments": int(len(subset)),
        "success_rate": round(len(good) / len(subset), 2),
        "good_scan_speeds_mm_s": sorted(good["scan_speed_mm_s"].tolist()),
        "laser_power_W": int(subset["laser_power_W"].iloc[0]),
    }

In [12]:
def make_proposal(x):
    """Build molar fractions for FeCoCrNiNbx (equal Fe, Co, Cr, Ni)."""
    base = 1 / (4 + x)
    return {"Fe": base, "Co": base, "Cr": base, "Ni": base, "Nb": x * base}

print(suggest_params(make_proposal(0.25), data))

{'requested_nb': 0.25, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}


In [13]:
proposals = [
    make_proposal(0.25),
    make_proposal(0.30),
    make_proposal(0.45),
    make_proposal(0.60),                                                  # outside the data range
    {"Fe": 0.3, "Co": 0.3, "Cr": 0.3, "Ni": 0.3, "Nb": 0.3},              # invalid: sum is 1.5
    {"Fe": 0.2, "Co": 0.2, "Cr": 0.2, "Ni": 0.2, "Ti": 0.2},              # invalid: Ti not allowed
]

plans = []
for i, proposal in enumerate(proposals, start=1):
    ok, message = validate(proposal)
    if not ok:
        print(f"Proposal {i} rejected: {message}")
        continue
    plan = {
        "proposal_id": i,
        "composition": proposal,
        "powder_mass_g_for_10g": powder_mass(proposal, 10),
        "suggestion": suggest_params(proposal, data),
    }
    plans.append(plan)
    print(f"Proposal {i}:", plan["suggestion"])

with open("experiment_plans.json", "w") as f:
    json.dump(plans, f, indent=2)
print(f"\n{len(plans)} plans saved to experiment_plans.json")

Proposal 1: {'requested_nb': 0.25, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}
Proposal 2: {'requested_nb': 0.3, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}
Proposal 3: {'requested_nb': 0.45, 'nearest_nb_in_data': 0.45, 'in_data_range': True, 'n_similar_experiments': 4, 'success_rate': 0.25, 'good_scan_speeds_mm_s': [4.0], 'laser_power_W': 280}
Proposal 4: {'requested_nb': 0.6, 'nearest_nb_in_data': 0.45, 'in_data_range': False, 'n_similar_experiments': 4, 'success_rate': 0.25, 'good_scan_speeds_mm_s': [4.0], 'laser_power_W': 280}
Proposal 5 rejected: Fractions sum to 1.500, not 1
Proposal 6 rejected: Element 'Ti' is not allowed

4 plans saved to experiment_plans.json


In [14]:
def log_result(proposal_id, nb_content, scan_speed_mm_s, outcome, notes="", path="results_log.csv"):
    """Append one experiment result to the log file."""
    if outcome not in ("good", "partial", "failed"):
        raise ValueError("outcome must be 'good', 'partial' or 'failed'")
    row = pd.DataFrame([{
        "proposal_id": proposal_id,
        "nb_content": nb_content,
        "scan_speed_mm_s": scan_speed_mm_s,
        "outcome": outcome,
        "notes": notes,
    }])
    row.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

In [15]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [16]:
def plan_for_nb(x: float) -> dict:
    """Validate the FeCoCrNiNbx composition for a given Nb ratio x, calculate
    the powder masses for a 10 g batch and suggest scan speeds from past experiments.

    Args:
        x: Nb ratio in FeCoCrNiNbx, a number greater than 0 (for example 0.3)
    """
    try:
        x = float(x)
    except (TypeError, ValueError):
        return {"error": f"x must be a number, got {x!r}"}
    if not 0 < x <= 1:
        return {"error": f"x must be between 0 and 1, got {x}. Call the tool again with the correct Nb ratio."}
    proposal = make_proposal(x)
    ok, msg = validate(proposal)
    if not ok:
        return {"error": msg}
    return {
        "powder_mass_g_for_10g": powder_mass(proposal, 10),
        "suggestion": suggest_params(proposal, data),
    }

print(plan_for_nb(0.3))
print(plan_for_nb(0))

{'powder_mass_g_for_10g': {'Fe': 2.204, 'Co': 2.326, 'Cr': 2.052, 'Ni': 2.317, 'Nb': 1.1}, 'suggestion': {'requested_nb': 0.3, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}}
{'error': 'x must be between 0 and 1, got 0.0. Call the tool again with the correct Nb ratio.'}


In [17]:
import ollama
client = ollama.Client(host="http://127.0.0.1:11434", trust_env=False)

r = client.chat(model="llama3.2",
                messages=[{"role": "user", "content": "Say hello in one sentence."}])
print(r.message.content)

Hello!


In [18]:
import re

def plan_for_composition(composition: str) -> dict:
    """Plan an experiment for a FeCoCrNiNbx alloy given its name, for example 'FeCoCrNiNb0.3'.
    Returns powder masses for a 10 g batch and scan speeds suggested from past experiments.

    Args:
        composition: alloy name, FeCoCrNiNb followed by the Nb ratio, for example 'FeCoCrNiNb0.3'
    """
    match = re.fullmatch(r"FeCoCrNiNb(\d*\.?\d+)", str(composition).strip())
    if not match:
        return {"error": f"Could not read the composition {composition!r}. Use the form 'FeCoCrNiNb0.3'."}
    return plan_for_nb(float(match.group(1)))

print(plan_for_composition("FeCoCrNiNb0.3"))
print(plan_for_composition("hello"))

{'powder_mass_g_for_10g': {'Fe': 2.204, 'Co': 2.326, 'Cr': 2.052, 'Ni': 2.317, 'Nb': 1.1}, 'suggestion': {'requested_nb': 0.3, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}}
{'error': "Could not read the composition 'hello'. Use the form 'FeCoCrNiNb0.3'."}


In [19]:
import json
import ollama

client = ollama.Client(host="http://127.0.0.1:11434", trust_env=False)

MODEL = "llama3.2"
tools = {"plan_for_composition": plan_for_composition}

messages = [{
    "role": "user",
    "content": "Call the plan_for_composition tool with composition 'FeCoCrNiNb0.3'. "
               "Then summarize the experiment plan using only the numbers returned by the tool."
}]

for step in range(5):
    response = client.chat(model=MODEL, messages=messages, tools=[plan_for_composition],
                           options={"temperature": 0})
    messages.append(response.message)
    if not response.message.tool_calls:
        print("FINAL ANSWER:\n", response.message.content)
        break
    for call in response.message.tool_calls:
        print("Tool call:", call.function.name, call.function.arguments)
        result = tools[call.function.name](**call.function.arguments)
        print("Tool result:", result)
        messages.append({
            "role": "tool",
            "tool_name": call.function.name,
            "content": json.dumps(result),
        })

Tool call: plan_for_composition {'composition': 'FeCoCrNiNb0.3'}
Tool result: {'powder_mass_g_for_10g': {'Fe': 2.204, 'Co': 2.326, 'Cr': 2.052, 'Ni': 2.317, 'Nb': 1.1}, 'suggestion': {'requested_nb': 0.3, 'nearest_nb_in_data': 0.25, 'in_data_range': True, 'n_similar_experiments': 6, 'success_rate': 1.0, 'good_scan_speeds_mm_s': [1.5, 2.1, 3.0, 3.33, 4.0, 4.0], 'laser_power_W': 280}}
FINAL ANSWER:
 The experiment plan for the composition 'FeCoCrNiNb0.3' includes:

- Powder mass of 10g for each element: Fe = 2.204g, Co = 2.326g, Cr = 2.052g, Ni = 2.317g, Nb = 1.1g
- Suggested laser power: 280W
- Suggested laser scan speeds: 1.5mm/s, 2.1mm/s, 3.0mm/s, 3.33mm/s, 4.0mm/s, 4.0mm/s
- Recommended number of similar experiments: 6
- Success rate: 100%
